# WTI Crude Oil — Protected Evaluation (Notebook 6 of 7)

> **Part 6 of 7.** Requires Notebook 5 to have been run first —
> the adaptive agent strategy variants must be trained.

This is the culminating comparison: all stateless predictors from Notebook 4
versus all three trained adaptive agent variants (plus the untrained baseline)
on the **held-out 2026 data**.

The evaluation period is Feb–Mar 2026 — the heart of the Persian Gulf
geopolitical price shock. Neither the stateless methods nor the adaptive agent
has seen this data. The question is whether the agent's 2025 training improved
its calibration for exactly the kind of regime it was trained on.

| | Stateless methods | Untrained agent | Trained agent variants |
|---|---|---|---|
| Training | None | None | 2025 curriculum (NB05) |
| Eval data | 2026 (never seen) | 2026 (never seen) | 2026 (never seen) |
| Strategy updates during eval | N/A | **Frozen** | **Frozen** |

Three trained variants are compared — one per NB05 training activity:
`wti-strategy-act1` (self-directed), `wti-strategy-stats` (stats curriculum),
`wti-strategy-news` (news curriculum). The untrained agent provides the
adaptive agent's own baseline — isolating the value of training from the
value of having an adaptive strategy at all.

---
## 0. Setup & Freeze

In [ ]:
import warnings
from pathlib import Path

import pandas as pd

from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
)
from aieng.forecasting.evaluation.backtest import BacktestResult
from energy_oil_forecasting.adaptive_agent import build_wti_adaptive_predictor
from energy_oil_forecasting.adaptive_agent.curriculum.snapshot_utils import (
    state_checksum,
)
from energy_oil_forecasting.analysis import score_backtest_results
from energy_oil_forecasting.data import build_wti_service

warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
_NB_DIR = Path('.')
_SKILLS_ROOT = _NB_DIR / 'adaptive_agent' / 'skills'
_CURRICULUM_DIR = _NB_DIR / 'adaptive_agent' / 'curriculum'
_SPECS_DIR = _NB_DIR / 'specs'

SEED_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy'       # untrained baseline
ACT1_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-act1'  # Act 1: self-directed
STATS_STRATEGY_DIR = _SKILLS_ROOT / 'wti-strategy-stats' # Act 2a: stats curriculum
NEWS_STRATEGY_DIR  = _SKILLS_ROOT / 'wti-strategy-news'  # Act 2b: news curriculum

# All four variants evaluated in Section 3:
ADAPTIVE_VARIANTS = {
    'Agent — untrained':    SEED_STRATEGY_DIR,
    'Agent — Act 1':        ACT1_STRATEGY_DIR,
    'Agent — Act 2a (stats)': STATS_STRATEGY_DIR,
    'Agent — Act 2b (news)':  NEWS_STRATEGY_DIR,
}

# ── Model ─────────────────────────────────────────────────────────────────────
AGENT_MODEL = 'gemini-3.5-flash'

# ── Run guard ─────────────────────────────────────────────────────────────────
RUN_EVAL = False   # Set True on first run; commit outputs; leave False.

# ── Data service ──────────────────────────────────────────────────────────────
data_service = build_wti_service()
print('Setup complete.')

In [ ]:
# ── Freeze: record pre-eval checksums for all adaptive variants ──────────────
# We verify post-eval that no skill state files were modified during eval.
_pre_eval_checksums = {name: state_checksum(d) for name, d in ADAPTIVE_VARIANTS.items()}
print('Pre-eval checksums recorded:')
for name, ck in _pre_eval_checksums.items():
    print(f'  {name}: {ck[:16]}...')

---
## 1. The Knowledge-Cutoff Teaching Point

**Gemini's parametric knowledge cutoff is approximately January 2025.**
This has a concrete implication for this evaluation:

- The **training period** (2025) is at or beyond the model's parametric
  knowledge horizon. During curriculum delivery in NB05, the agent could not
  rely on memorized facts about 2025 WTI prices — it had to reason from the
  backtest report and pre-cached news summaries we provided.

- The **evaluation period** (Feb–Mar 2026) is definitively post-cutoff.
  During eval, the agent must rely entirely on:
  1. Its Google Search tool (with `cutoff_date` enforcement per origin)
  2. Its code execution tool (for statistical analysis of available data)
  3. Its accumulated strategy state (calibration corrections from training)

This is a clean test of what the training phase actually adds: it cannot be
attributed to the model's parametric knowledge of the eval period.

---
## 2. Load Stateless Eval Results

Notebook 4 saved the 2026 eval results for the top stateless predictors.
We load them here — no re-run needed.

In [ ]:
# ── Load eval results from NB04 ─────────────────────────────────────────────
# NB04 saves stateless results with predictor display names as the stem prefix
# (e.g. eval_AutoARIMA.json, eval_Naive_(Last_Value).json).
# We load only those — agent results are handled separately in Section 3.
_STATELESS_NAMES = ['AutoARIMA', 'Naive_(Last_Value)', 'Naive (Last Value)']
_stateless_jsons = [
    f for f in sorted(_CURRICULUM_DIR.glob('eval_*.json'))
    if not f.stem.removeprefix('eval_').startswith('Agent')
]
if not _stateless_jsons:
    raise FileNotFoundError(
        'No stateless eval result files found in adaptive_agent/curriculum/. '
        'Run 04_systematic_backtest_eval.ipynb first.'
    )

all_eval_results: dict[str, BacktestResult] = {}
for f in _stateless_jsons:
    name = f.stem.removeprefix('eval_').replace('_', ' ')
    all_eval_results[name] = BacktestResult.model_validate_json(f.read_text())

print(f'Loaded {len(all_eval_results)} stateless eval result(s):')
for name, r in all_eval_results.items():
    print(f'  {name}: {len(r.predictions)} predictions, '
          f'mean CRPS = {r.mean_crps:.4f}')

---
## 3. Run Adaptive Agent Variants on Eval Spec

Each adaptive agent variant is evaluated on the same 2026 eval spec  
(`energy_oil_eval.yaml`) used by the stateless predictors in NB04.

> **Run guard:** `RUN_EVAL = False` by default. Set to `True` on first run,
> commit the saved result files, and leave `False` for reproducibility.

In [ ]:
import yaml  # noqa: PLC0415
with open(_SPECS_DIR / 'energy_oil_eval.yaml') as _f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(_f))

def _safe_key(name: str) -> str:
    return name.replace(' ', '_').replace('(', '').replace(')', '').replace('—', '').strip('_')

if RUN_EVAL:
    print('Running all adaptive agent variants on 2026 eval spec...')
    print('(Live API calls — first run may take several minutes.)\n')

    for variant_name, strategy_dir in ADAPTIVE_VARIANTS.items():
        predictor = build_wti_adaptive_predictor(strategy_dir=strategy_dir, model=AGENT_MODEL)
        result_dict = cached_multi_backtest(predictor, eval_spec, data_service)
        result = next(iter(result_dict.values()))
        all_eval_results[variant_name] = result
        safe = _safe_key(variant_name)
        (_CURRICULUM_DIR / f'eval_{safe}.json').write_text(
            result.model_dump_json(), encoding='utf-8'
        )
        print(f'  {variant_name}: mean CRPS = {result.mean_crps:.4f} ✓')

    print('\nEval complete.')
else:
    # Load committed adaptive eval results if present
    for variant_name in ADAPTIVE_VARIANTS:
        safe = _safe_key(variant_name)
        _f = _CURRICULUM_DIR / f'eval_{safe}.json'
        if _f.exists():
            all_eval_results[variant_name] = BacktestResult.model_validate_json(
                _f.read_text()
            )
    print('RUN_EVAL = False — using committed outputs (or set True to re-run).')
    print(f'Eval results available: {list(all_eval_results)}')

---
## 4. Comparative Scorecard

All predictors evaluated on the same 8 weekly origins in early 2026 — a period of
major geopolitical volatility that drove WTI from ~$65 to above $100.

**Key comparisons:**
- **Untrained vs AutoARIMA**: value of adding live news search, even without training
- **Act 2b vs Untrained**: value of training with news-grounded curriculum
- **Act 2a vs Act 2b**: marginal effect of including news context during training
- **All agents vs stateless**: value of the adaptive architecture overall

In [ ]:
scorecard_rows = []
for name, result in all_eval_results.items():
    # score_backtest_results expects {task_id: BacktestResult}; wrap single results
    _result_for_scoring = result if isinstance(result, dict) else {name: result}
    scores = score_backtest_results(_result_for_scoring, data_service)
    scorecard_rows.append(
        {
            'Predictor': name,
            'Mean CRPS': round(scores.get('mean_crps', float('nan')), 3),
            'MAE h=21d': round(scores.get('mae_h21', float('nan')), 3),
            '80% CI Coverage': f"{scores.get('coverage_80', float('nan')):.1f}%",
        }
    )

df_scorecard = pd.DataFrame(scorecard_rows).set_index('Predictor')
df_scorecard = df_scorecard.sort_values('Mean CRPS')

print('━' * 72)
print('2026 PROTECTED EVAL — ALL PREDICTORS (sorted by CRPS, lower is better):')
print('━' * 72)
print(df_scorecard.to_string())


---
## 5. Forecast Comparison — Selected Origins

Point forecasts and 80% prediction intervals at two contrasting origins:
one before the price shock began, one during it.  
Each panel shows the 5, 10, and 21-day-ahead forecasts from every method
alongside the realised price.

In [ ]:
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from aieng.forecasting.evaluation.prediction import ContinuousForecast

_full_series = data_service.get_series('wti_crude_oil_price', as_of=datetime.now())
_price_ts = pd.to_datetime(_full_series['timestamp'])
_price_vals = _full_series['value'].values

# Collect all predictions into a flat lookup: (predictor, as_of_date) -> list[Prediction]
_preds_by_origin: dict[tuple[str, str], list] = {}
for name, result in all_eval_results.items():
    for pred in result.predictions:
        key = (name, str(pred.as_of.date()))
        _preds_by_origin.setdefault(key, []).append(pred)

# Get all origins and pick two representative ones
_origins = sorted({str(pred.as_of.date()) for result in all_eval_results.values() for pred in result.predictions})
print(f'Eval origins: {_origins}')

# Use first and last origin for contrast
_SHOW_ORIGINS = [_origins[0], _origins[-1]] if len(_origins) >= 2 else _origins

_COLORS = {
    'Naive (Last Value)': 'gray',
    'AutoARIMA': 'steelblue',
    'Agent — untrained': '#f4a261',
    'Agent — Act 1': '#e76f51',
    'Agent — Act 2a (stats)': '#2a9d8f',
    'Agent — Act 2b (news)': '#6a0572',
}

fig = make_subplots(
    rows=1, cols=len(_SHOW_ORIGINS),
    subplot_titles=[f'Origin: {o}' for o in _SHOW_ORIGINS],
    shared_yaxes=True,
)

for col_idx, origin in enumerate(_SHOW_ORIGINS, 1):
    origin_ts = pd.Timestamp(origin)
    # Price context: 30 days before to 25 days after
    ctx_mask = (_price_ts >= origin_ts - pd.Timedelta(days=30)) & (_price_ts <= origin_ts + pd.Timedelta(days=25))
    ctx_dates = _price_ts[ctx_mask]
    ctx_prices = _price_vals[ctx_mask]

    # Actual price line
    fig.add_trace(go.Scatter(
        x=ctx_dates, y=ctx_prices,
        mode='lines', name='Actual' if col_idx == 1 else None,
        line=dict(color='black', width=2),
        showlegend=(col_idx == 1),
    ), row=1, col=col_idx)

    # Origin marker
    fig.add_vline(x=origin_ts.timestamp() * 1000, line_dash='dash',
                  line_color='black', opacity=0.4, row=1, col=col_idx)

    for name, color in _COLORS.items():
        preds = _preds_by_origin.get((name, origin), [])
        if not preds:
            continue
        for pred in sorted(preds, key=lambda p: p.forecast_date):
            if not isinstance(pred.payload, ContinuousForecast):
                continue
            fc_date = pd.Timestamp(pred.forecast_date)
            pt = pred.payload.point_forecast
            lo = pred.payload.quantiles.get(0.1, pt)
            hi = pred.payload.quantiles.get(0.9, pt)
            fig.add_trace(go.Scatter(
                x=[fc_date], y=[pt],
                mode='markers',
                marker=dict(color=color, size=8, symbol='diamond'),
                name=name if col_idx == 1 else None,
                showlegend=(col_idx == 1 and pred == preds[0]),
                legendgroup=name,
            ), row=1, col=col_idx)
            fig.add_trace(go.Scatter(
                x=[fc_date, fc_date], y=[lo, hi],
                mode='lines',
                line=dict(color=color, width=3),
                showlegend=False,
                legendgroup=name,
            ), row=1, col=col_idx)

fig.update_layout(
    title='Point forecasts (diamond) and 80% CI (bar) by predictor — 2026 eval origins',
    height=500, width=1000,
    legend=dict(orientation='h', y=-0.2),
    yaxis_title='WTI price (USD/bbl)',
)
fig.show()

---
## 6. Agent Rationale — What the Agents Said

Each adaptive agent records its reasoning in the prediction metadata.
Below are the `agent_rationale` strings from the first eval origin for
each trained variant — showing concretely how the training experience
shaped what the agent attends to.

Compare the untrained agent (which has the same architecture and news access)
against the trained variants to see how curriculum learning changes the
agent's framing and confidence.

In [ ]:
from IPython.display import Markdown, display

_RATIONALE_VARIANTS = [
    'Agent — untrained',
    'Agent — Act 2a (stats)',
    'Agent — Act 2b (news)',
]
_first_origin = sorted(
    {str(p.as_of.date()) for p in next(iter(all_eval_results.values())).predictions}
)[0]

for name in _RATIONALE_VARIANTS:
    if name not in all_eval_results:
        continue
    preds = [
        p for p in all_eval_results[name].predictions
        if str(p.as_of.date()) == _first_origin
    ]
    if not preds:
        continue
    rationale = preds[0].metadata.get('agent_rationale', '*(no rationale stored)*')
    display(Markdown(
        f'### {name}\n'
        f'*Origin: {_first_origin}*\n\n'
        f'> {rationale.strip()}'
    ))
    print()

---
## 7. A Note on Knowledge Cutoff and Data Leakage

The Act 2b (news) training curriculum used pre-cached weekly WTI news summaries
from 2025, generated by `scripts/cache_wti_curriculum_news.py` with strict
temporal cutoff enforcement at each search date.

However, there is an important limitation to acknowledge:

> **The summaries were generated by an LLM that may have encoded parametric
> knowledge of 2025 events**, not just information from a web search at that
> date. This is a subtle but real form of data leakage — the 'pre-cached news'
> may contain implicit future knowledge that a genuine real-time search at that
> date would not have.

This does not invalidate the experiment, but it does mean the Act 2b improvement
should be interpreted carefully:

- It shows that *news-grounded training context* improves calibration.
- It does not definitively show that *real-time news retrieval* alone explains
  the improvement, since the training signal may have been enriched by
  parametric model knowledge of 2025 outcomes.

**To eliminate this concern**: re-run `cache_wti_curriculum_news.py` using a model
whose knowledge cutoff predates the search dates, or verify the summaries manually
against contemporaneous sources for a sample of dates.

This is a live research question — and a great discussion topic for bootcamp
participants thinking about how to evaluate agentic systems rigorously.

---
## 8. Freeze Verification

Confirm that the evaluation did not trigger any skill state mutations.
The checksums should match the pre-eval values recorded in Setup.

In [ ]:
print('State integrity check (all variants should be unchanged):')
all_ok = True
for name, d in ADAPTIVE_VARIANTS.items():
    ck_after = state_checksum(d)
    ok = ck_after == _pre_eval_checksums[name]
    all_ok = all_ok and ok
    print(f'  {name}: {"✓ unchanged" if ok else "⚠ MODIFIED"}')

if not all_ok:
    print('\nWarning: at least one agent updated its strategy during evaluation.')
    print('See the closing note for how to explore this intentionally.')

---
## 9. Closing Note — Unfreezing

The adaptive agent evaluated here was **frozen**: its strategy state was not
updated during evaluation. This gives a clean before/after comparison between
trained and stateless predictors on identical eval origins.

But in live deployment, you would not freeze the agent. After each resolved
prediction, you would send a resolution message and let the agent decide whether
to record an observation or update a hypothesis. Over time, the strategy evolves.

**To explore unfreezing:**

1. Set `RUN_EVAL = True`.
2. Remove the state checksum assertion (or ignore the warning).
3. Modify the eval loop to send a resolution message after each prediction:

```python
# After each prediction resolves:
resolution_msg = (
    f'The actual WTI price on {pred.forecast_date.date()} was {actual:.2f}. '
    f'Your point forecast was {pred.payload.point_forecast:.2f} '
    f'(error: {pred.payload.point_forecast - actual:+.2f}). '
    'Please review whether this outcome is relevant to any open hypothesis.'
)
await runner.run_text_async(resolution_msg)
```

4. Re-run and compare the final strategy state to the frozen baseline.

Notebook 7 shows how to do this interactively via `adk web`.